# Inference

In [ ]:
import logging
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from typing import Dict, List, Union

import numpy as np
import pandas as pd
from openai import OpenAI, AzureOpenAI
from scipy import stats
from sklearn.model_selection import KFold
from tqdm import tqdm

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(message)s")
logger = logging.getLogger(__name__)



def get_model_prediction(clinical_note: str, task: str, client, model_name: str) -> str:
    """Get model prediction for a single example."""

    # Select appropriate prompt template based on task
    if task == "tumor_type":
        prompt = f"""You are an expert neuro-radiologist. Based on the following clinical notes, classify the brain tumor type into one of these categories: glioblastoma, astrocytoma, oligodendroglioma, or oligoastrocytoma.

Clinical Notes:
{clinical_note}

Provide your classification in the following format:
CLASSIFICATION: [tumor type]

Only output one of the four specified categories."""

    elif task == "survival":
        prompt = f"""You are an expert neuro-oncologist. Based on the following clinical notes, predict whether the patient will survive beyond 1 year from diagnosis.

Clinical Notes:
{clinical_note}

Provide your prediction in the following format:
PREDICTION: [SURVIVE/NOT_SURVIVE]"""

    else:  # mgmt
        prompt = f"""You are an expert molecular pathologist. Based on the following clinical notes, predict whether the patient's tumor has MGMT promoter methylation (positive) or not (negative).

Clinical Notes:
{clinical_note}

Provide your prediction in the following format:
MGMT_STATUS: [POSITIVE/NEGATIVE]"""

    # Add retry logic with exponential backoff
    max_retries = 3
    for attempt in range(max_retries):
        try:
            completion = client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,  # Use deterministic sampling
                max_tokens=50,
            )
            return completion.choices[0].message.content
        except Exception as e:
            if attempt == max_retries - 1:
                logger.error(f"Failed after {max_retries} attempts: {e}")
                return ""
            time.sleep(2**attempt)  # Exponential backoff


def parse_prediction(prediction: str, task: str) -> int:
    """Parse model output into numerical labels."""
    prediction = prediction.lower().strip()

    if task == "tumor_type":
        if "glioblastoma" in prediction:
            return 0
        elif "astrocytoma" in prediction:
            return 1
        elif "oligodendroglioma" in prediction:
            return 2
        elif "oligoastrocytoma" in prediction:
            return 3
        return -1

    elif task == "survival":
        if "survive" in prediction:
            return 1
        elif "not_survive" in prediction:
            return 0
        return -1

    else:  # mgmt
        if "positive" in prediction:
            return 1
        elif "negative" in prediction:
            return 0
        return -1


def process_batch(args: tuple) -> List[int]:
    """Process a batch of examples."""
    clinical_notes, task, client, model_name = args
    predictions = []

    for note in clinical_notes:
        prediction = get_model_prediction(note, task, client, model_name)
        label = parse_prediction(prediction, task)
        predictions.append(label)

    return predictions


def get_predictions(df: pd.DataFrame, task: str, client, model_name: str, batch_size: int = 10) -> np.ndarray:
    """Get predictions for all examples in parallel batches."""
    # Use 'report' column for tumor_df and 'text' column for mgmt_df
    if "report" in df.columns:
        clinical_notes = df["report"].tolist()
    elif "text" in df.columns:
        clinical_notes = df["text"].tolist()
    elif "Anon Report Text" in df.columns:
        clinical_notes = df["Anon Report Text"].tolist()
    else:
        raise ValueError("No valid text column found in dataframe")

    batches = [
        clinical_notes[i : i + batch_size]
        for i in range(0, len(clinical_notes), batch_size)
    ]

    logging.getLogger("openai").setLevel(logging.WARNING)
    logging.getLogger("httpx").setLevel(logging.WARNING)

    predictions = []
    with ThreadPoolExecutor(max_workers=4) as executor:
        args = [(batch, task, client, model_name) for batch in batches]
        for batch_predictions in tqdm(
            executor.map(process_batch, args),
            total=len(batches),
            desc=f"Getting {task} predictions",
        ):
            predictions.extend(batch_predictions)

    return np.array(predictions)


def calculate_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict:
    """Calculate evaluation metrics."""
    from sklearn.metrics import (
        accuracy_score,
        precision_recall_fscore_support,
        roc_auc_score,
    )

    metrics = {}
    metrics["accuracy"] = accuracy_score(y_true, y_pred)
    precision, recall, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    metrics["precision"] = precision
    metrics["recall"] = recall
    metrics["f1_macro"] = f1_macro

    # Calculate AUC for binary and multiclass
    if len(np.unique(y_true)) == 2:
        metrics["aucroc"] = roc_auc_score(y_true, y_pred)
    else:
        y_pred_onehot = np.eye(len(np.unique(y_true)))[y_pred]
        metrics["aucroc"] = roc_auc_score(y_true, y_pred_onehot, multi_class="ovr")

    return metrics


def format_results_table(results: Dict[str, Dict], task: str, model_name: str) -> str:
    """Format results as a markdown table."""
    metric_order = ["accuracy", "precision", "recall", "f1_macro", "aucroc"]
    metric_display = {
        "accuracy": "Accuracy",
        "precision": "Precision",
        "recall": "Recall",
        "f1_macro": "F1-macro",
        "aucroc": "AUCROC",
    }

    headers = ["Model"] + [f"{metric_display[m]} (%)" for m in metric_order]

    table = ["|" + "|".join(headers) + "|"]
    table.append("|" + "|".join(["---"] * len(headers)) + "|")

    # Calculate mean and std for each metric
    row = [model_name]
    for metric in metric_order:
        values = [fold_results[metric] for fold_results in results.values()]
        mean_value = np.mean(values) * 100
        std_value = np.std(values) * 100
        row.append(f"{mean_value:.1f} ± {std_value:.1f}")

    table.append("|" + "|".join(row) + "|")

    return "\n".join(table)


def create_local_client(base_url="http://localhost:8000/v1", api_key="token-abc123"):
    """Create a local OpenAI client for BioMistral."""
    return OpenAI(
        base_url=base_url,
        api_key=api_key,
    )


def create_azure_client(api_key, api_version, azure_endpoint):
    """Create an Azure OpenAI client."""
    return AzureOpenAI(
        api_key=api_key,
        api_version=api_version,
        azure_endpoint=azure_endpoint,
    )


# Configuration - choose one of these client setups
# For BioMistral (local)
#MODEL_NAME = "BioMistral-7B"
#client = create_local_client(
#    base_url="http://localhost:8000/v1",
#    api_key="token-abc123",
#)

# For Azure OpenAI (uncomment to use)
MODEL_NAME = "gpt-4o-2024-08-06"
client = create_azure_client(
    api_key = "ZTIzNjBlMWNjNGU4NDllNzhkMDg0OGFjYTg0OTY2YzE6OTNlMzVlRjMxZmRCNEMxQThmQTcyRkZjQzc5N0Q0Yzg=", 
    api_version="2024-10-21",
    azure_endpoint="https://unified-api.ucsf.edu/general",
)


def main():
    # Load datasets
    tumor_df = pd.read_csv("data/eval/tumor_type_eval.csv")  # Uses 'report' column
    mgmt_df = pd.read_csv("data/eval/mgmt_eval.csv")  # Uses 'text' column
    os_df = pd.read_csv("data/eval/os_eval.csv")
    mgmt_df["mgmt_label"] = (
        mgmt_df["mgmt"].notna() & (mgmt_df["mgmt"] == "positive")
    ).astype(int)

    # Create results directory
    results_dir = Path(f"results/{MODEL_NAME}")
    results_dir.mkdir(exist_ok=True, parents=True)

    # Process each task
    tasks = {
        "tumor_type": {
            "df": tumor_df,
            "label_col": "class_label",
        },
        "survival": {
            "df": os_df.copy(),
            "label_col": "one_year_survival",
        },
        "mgmt": {
            "df": mgmt_df.copy(),
            "label_col": "mgmt_label",
        },
    }

    kf = KFold(n_splits=10, shuffle=True, random_state=42)
    all_results = []

    for task_name, task_config in tasks.items():
        logger.info(f"\nProcessing {task_name} task...")

        df = task_config["df"]
        valid_mask = df[task_config["label_col"]].notna()
        df = df[valid_mask]

        fold_results = {}
        for fold_idx, (train_idx, test_idx) in enumerate(kf.split(df), 1):
            logger.info(f"\nProcessing fold {fold_idx}...")

            test_df = df.iloc[test_idx]

            # Get predictions for test set
            predictions = get_predictions(test_df, task_name, client, MODEL_NAME)
            true_labels = test_df[task_config["label_col"]].values

            # Calculate metrics for this fold
            fold_results[fold_idx] = calculate_metrics(true_labels, predictions)

        # Format results table with mean and std
        results_table = format_results_table(fold_results, task_name, MODEL_NAME)
        all_results.append(f"# {task_name.title()} Task Results\n\n{results_table}\n")

        logger.info(f"\nResults for {task_name} task:")
        logger.info("\n" + results_table)

    output_file = results_dir / "all_results.md"
    with open(output_file, "w") as f:
        f.write("\n".join(all_results))

    logger.info(f"\nAll results saved to {output_file}")


if __name__ == "__main__":
    main()
